# NCBI GEO 유전자 발현 바이오 데이터 전처리 실습

**데이터**: GSM827073 (Agilent 1-color microarray)  
**출처**: https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM827073  

본 노트북은 Agilent Feature Extraction 압축 파일(`*.txt.gz`)을 **먼저 압축 해제한 뒤**  
FEATURES 영역을 추출하고, 생물학적 probe만 남긴 뒤 품질·발현 필터와 log2 변환을 적용하여 머신러닝용 CSV를 생성합니다.

In [1]:
# ============================================================
# 0. 라이브러리 불러오기 및 경로 설정
# ============================================================
import gzip
import shutil
from pathlib import Path

import numpy as np
import pandas as pd

# 데이터 파일 경로 (노트북과 동일 폴더에 위치)
DATA_DIR = Path(".")
GZ_PATH = DATA_DIR / "GSM827073_US10283824_252800412039_S01_GE1_107_Sep09_1_2.txt.gz"
TXT_PATH = DATA_DIR / "GSM827073_US10283824_252800412039_S01_GE1_107_Sep09_1_2.txt"  # 압축 해제 결과
CSV_OUT = DATA_DIR / "GSM827073_preprocessed.csv"

# ------------------------------------------------------------
# 압축 파일(.txt.gz)을 풀어 TXT로 저장한 뒤 이후 단계에서 사용
# ------------------------------------------------------------
if not GZ_PATH.exists():
    raise FileNotFoundError(f"압축 파일을 찾을 수 없습니다: {GZ_PATH.resolve()}")

print(f"[0] 압축 파일: {GZ_PATH.name}")
print(f"[0] 압축 해제 대상: {TXT_PATH.name}")

# gzip → txt 바이너리 복사 (기존 txt가 있어도 원본 gz 기준으로 다시 생성)
with gzip.open(GZ_PATH, "rb") as f_in, open(TXT_PATH, "wb") as f_out:
    shutil.copyfileobj(f_in, f_out)

print(f"[0] 압축 해제 완료: {TXT_PATH.resolve()}")
print(f"[0] 해제된 파일 크기: {TXT_PATH.stat().st_size:,} bytes")

# 단계별 데이터 크기를 기록해 마지막에 전후 비교에 사용
step_sizes = {}  # {단계명: 행(probe) 개수}


def report_size(step_name: str, df: pd.DataFrame) -> None:
    """각 전처리 단계의 행/열 크기를 출력하고 기록한다."""
    n_rows, n_cols = df.shape
    step_sizes[step_name] = n_rows
    print(f"[{step_name}] 데이터 크기: {n_rows:,} rows × {n_cols} columns")

[0] 압축 파일: GSM827073_US10283824_252800412039_S01_GE1_107_Sep09_1_2.txt.gz
[0] 압축 해제 대상: GSM827073_US10283824_252800412039_S01_GE1_107_Sep09_1_2.txt
[0] 압축 해제 완료: C:\MyCursorLab\01_바이오 데이터 이해 및 LLM 기반 전처리\GSM827073_US10283824_252800412039_S01_GE1_107_Sep09_1_2.txt
[0] 해제된 파일 크기: 9,715,524 bytes


## 1. TXT 파일을 Pandas로 읽기

Agilent Feature Extraction 파일은 FEPARAMS / STATS / FEATURES 등 여러 섹션이 섞여 있습니다.  
앞에서 압축 해제한 **TXT 파일**에서 FEATURES 헤더 위치를 찾은 뒤, 해당 위치부터 Pandas로 읽습니다.

In [2]:
# ------------------------------------------------------------
# 1) 압축 해제한 TXT에서 FEATURES 섹션 시작 줄 탐색
# ------------------------------------------------------------
features_skip = None

# Agilent Feature Extraction 파일은 일부 비-UTF8 바이트가 포함될 수 있어
# latin-1(ISO-8859-1)로 읽으면 디코딩 오류 없이 안전하게 파싱됨
with open(TXT_PATH, "r", encoding="latin-1") as f:
    for i, line in enumerate(f):
        # FEATURES로 시작하는 줄이 컬럼 헤더
        if line.startswith("FEATURES"):
            features_skip = i
            break

if features_skip is None:
    raise ValueError("FEATURES 섹션을 찾지 못했습니다. 파일 형식을 확인하세요.")

print(f"[1] 사용 파일: {TXT_PATH.name} (압축 해제본)")
print(f"[1] FEATURES 헤더 위치: {features_skip}번째 줄 (0-based)")

# ------------------------------------------------------------
# 1-2) Pandas로 FEATURES 이후 전체를 읽기
#   - 압축 해제한 TXT를 직접 읽음 (gzip 아님)
#   - sep='\t' : 탭 구분
#   - encoding='latin-1' : 특수문자 디코딩 오류 방지
# ------------------------------------------------------------
raw = pd.read_csv(
    TXT_PATH,
    sep="\t",
    skiprows=features_skip,
    encoding="latin-1",
    low_memory=False,
)

print("[1] TXT 파일 읽기 완료")
report_size("1_raw_read", raw)
display(raw.head(3))

[1] 사용 파일: GSM827073_US10283824_252800412039_S01_GE1_107_Sep09_1_2.txt (압축 해제본)
[1] FEATURES 헤더 위치: 9번째 줄 (0-based)
[1] TXT 파일 읽기 완료
[1_raw_read] 데이터 크기: 62,976 rows × 26 columns


,FEATURES,FeatureNum,Row,Col,SubTypeMask,ControlType,ProbeName,SystematicName,PositionX,PositionY,...,gIsFeatNonUnifOL,gIsBGNonUnifOL,gIsFeatPopnOL,gIsBGPopnOL,IsManualFlag,gBGSubSignal,gIsPosAndSignif,gIsWellAboveBG,SpotExtentX,gBGMeanSignal
0,DATA,1,1,1,260,1,GE_BrightCorner,GE_BrightCorner,5888.06,435.779,...,0,0,1,1,0,88326.80000,1,1,27.0811,29.1226
1,DATA,2,1,2,66,1,DarkCorner,DarkCorner,5909.40,435.586,...,0,0,0,1,0,-2.70684,0,0,27.5010,29.6418
2,DATA,3,1,3,66,1,DarkCorner,DarkCorner,5930.12,436.000,...,0,0,0,1,0,-1.81411,0,0,23.6960,29.5886


## 2. FEATURES 영역을 DataFrame으로 변환

첫 번째 컬럼(`FEATURES`)은 섹션 마커이며, 실제 probe 행은 값이 `DATA`인 행입니다.  
마커 컬럼을 제거하고 DATA 행만 남깁니다.

In [3]:
# ------------------------------------------------------------
# 2) FEATURES → 분석용 DataFrame 정리
# ------------------------------------------------------------
# 첫 컬럼명은 'FEATURES'이며, 각 행의 값은 'DATA' 또는 기타 마커
marker_col = raw.columns[0]
print(f"[2] 섹션 마커 컬럼명: {marker_col}")
print(f"[2] 마커 값 분포:\n{raw[marker_col].value_counts(dropna=False)}")

# DATA 행만 추출 (실제 feature/probe 측정값)
features = raw[raw[marker_col] == "DATA"].copy()

# 마커 컬럼 제거 후 인덱스 재설정
features = features.drop(columns=[marker_col]).reset_index(drop=True)

# 주요 수치/플래그 컬럼을 숫자형으로 변환 (이후 필터링을 위함)
numeric_cols = [
    "ControlType",
    "gProcessedSignal",
    "gIsPosAndSignif",
    "gIsWellAboveBG",
]
for col in numeric_cols:
    features[col] = pd.to_numeric(features[col], errors="coerce")

print("[2] FEATURES 영역 DataFrame 변환 완료")
report_size("2_features_df", features)
display(features.head(3))

[2] 섹션 마커 컬럼명: FEATURES
[2] 마커 값 분포:
FEATURES
DATA    62976
Name: count, dtype: int64
[2] FEATURES 영역 DataFrame 변환 완료
[2_features_df] 데이터 크기: 62,976 rows × 25 columns


,FeatureNum,Row,Col,SubTypeMask,ControlType,ProbeName,SystematicName,PositionX,PositionY,gProcessedSignal,...,gIsFeatNonUnifOL,gIsBGNonUnifOL,gIsFeatPopnOL,gIsBGPopnOL,IsManualFlag,gBGSubSignal,gIsPosAndSignif,gIsWellAboveBG,SpotExtentX,gBGMeanSignal
0,1,1,1,260,1,GE_BrightCorner,GE_BrightCorner,5888.06,435.779,86629.190000,...,0,0,1,1,0,88326.80000,1,1,27.0811,29.1226
1,2,1,2,66,1,DarkCorner,DarkCorner,5909.40,435.586,4.210043,...,0,0,0,1,0,-2.70684,0,0,27.5010,29.6418
2,3,1,3,66,1,DarkCorner,DarkCorner,5930.12,436.000,4.225198,...,0,0,0,1,0,-1.81411,0,0,23.6960,29.5886


## 3. 전체 Feature 수 확인

In [4]:
# ------------------------------------------------------------
# 3) 전체 Feature(probe) 개수
# ------------------------------------------------------------
n_features_total = len(features)
print(f"[3] 전체 Feature 수: {n_features_total:,}")
report_size("3_all_features", features)

[3] 전체 Feature 수: 62,976
[3_all_features] 데이터 크기: 62,976 rows × 25 columns


## 4. ControlType별 Feature 개수 확인

- `ControlType == 0` : 생물학적(biological) probe  
- `ControlType != 0` : 양성/음성 대조군, 코너 마커 등 컨트롤 probe

In [5]:
# ------------------------------------------------------------
# 4) ControlType별 개수
# ------------------------------------------------------------
control_counts = features["ControlType"].value_counts(dropna=False).sort_index()
print("[4] ControlType별 Feature 개수:")
print(control_counts.to_string())
print()
print(f"[4] ControlType==0 (biological): {int((features['ControlType'] == 0).sum()):,}")
print(f"[4] ControlType!=0 (control)   : {int((features['ControlType'] != 0).sum()):,}")
report_size("4_before_control_filter", features)

[4] ControlType별 Feature 개수:
ControlType
-1      308
 0    58717
 1     3951

[4] ControlType==0 (biological): 58,717
[4] ControlType!=0 (control)   : 4,259
[4_before_control_filter] 데이터 크기: 62,976 rows × 25 columns


## 5. ControlType이 0인 biological probe만 추출

In [6]:
# ------------------------------------------------------------
# 5) biological probe만 남기기
# ------------------------------------------------------------
bio = features[features["ControlType"] == 0].copy()
print("[5] ControlType==0 인 biological probe만 추출")
report_size("5_biological_probes", bio)

[5] ControlType==0 인 biological probe만 추출
[5_biological_probes] 데이터 크기: 58,717 rows × 25 columns


## 6. gProcessedSignal 결측 확인

`gProcessedSignal`은 Agilent에서 배경 보정·정규화된 발현값입니다. 결측이 있으면 이후 분석에서 제외합니다.

In [7]:
# ------------------------------------------------------------
# 6) gProcessedSignal 결측 확인 및 제거
# ------------------------------------------------------------
n_missing = bio["gProcessedSignal"].isna().sum()
print(f"[6] gProcessedSignal 결측 개수: {n_missing:,}")

bio_nomiss = bio.dropna(subset=["gProcessedSignal"]).copy()
print("[6] 결측 행 제거 후")
report_size("6_drop_missing_signal", bio_nomiss)

[6] gProcessedSignal 결측 개수: 0
[6] 결측 행 제거 후
[6_drop_missing_signal] 데이터 크기: 58,717 rows × 25 columns


## 7. gIsPosAndSignif == 1 인 Probe 추출

신호가 양수이며 통계적으로 유의한(positive and significant) probe만 남깁니다.

In [8]:
# ------------------------------------------------------------
# 7) Positive & Significant 플래그 필터
# ------------------------------------------------------------
pos_sig = bio_nomiss[bio_nomiss["gIsPosAndSignif"] == 1].copy()
print("[7] gIsPosAndSignif == 1 인 Probe 추출")
report_size("7_pos_and_signif", pos_sig)

[7] gIsPosAndSignif == 1 인 Probe 추출
[7_pos_and_signif] 데이터 크기: 53,565 rows × 25 columns


## 8. gIsWellAboveBG == 1 인 Probe 추출

배경(background)보다 충분히 높은 신호만 남깁니다.

In [9]:
# ------------------------------------------------------------
# 8) Well Above Background 플래그 필터
# ------------------------------------------------------------
above_bg = pos_sig[pos_sig["gIsWellAboveBG"] == 1].copy()
print("[8] gIsWellAboveBG == 1 인 Probe 추출")
report_size("8_well_above_bg", above_bg)

[8] gIsWellAboveBG == 1 인 Probe 추출
[8_well_above_bg] 데이터 크기: 45,780 rows × 25 columns


## 9. 발현값이 낮은 Probe 필터링

품질 통과 probe 중에서도 발현이 매우 낮은 경우 잡음에 가깝습니다.  
여기서는 **하위 25분위수(Q1) 이하** 발현값을 제거합니다.  
(필요 시 `LOW_EXPR_QUANTILE` 값을 바꿔 임계값을 조정할 수 있습니다.)

In [10]:
# ------------------------------------------------------------
# 9) 저발현 Probe 필터링 (하위 분위수 이하 제거)
# ------------------------------------------------------------
LOW_EXPR_QUANTILE = 0.25  # 하위 25% 제거

low_threshold = above_bg["gProcessedSignal"].quantile(LOW_EXPR_QUANTILE)
print(f"[9] 저발현 임계값 (quantile={LOW_EXPR_QUANTILE}): {low_threshold:.6g}")

expr_filtered = above_bg[above_bg["gProcessedSignal"] > low_threshold].copy()
print("[9] 저발현 Probe 필터링 완료")
report_size("9_low_expression_filtered", expr_filtered)

[9] 저발현 임계값 (quantile=0.25): 104.584
[9] 저발현 Probe 필터링 완료
[9_low_expression_filtered] 데이터 크기: 34,335 rows × 25 columns


## 10. SystematicName과 gProcessedSignal만 남기기

머신러닝/다운스트림 분석에 필요한 식별자(유전자/전사체 ID)와 발현값만 유지합니다.

In [11]:
# ------------------------------------------------------------
# 10) 분석에 필요한 컬럼만 선택
# ------------------------------------------------------------
slim = expr_filtered[["SystematicName", "gProcessedSignal"]].copy()
print("[10] SystematicName, gProcessedSignal 컬럼만 유지")
report_size("10_selected_columns", slim)
display(slim.head(5))

[10] SystematicName, gProcessedSignal 컬럼만 유지
[10_selected_columns] 데이터 크기: 34,335 rows × 2 columns


,SystematicName,gProcessedSignal
3,NM_144987,307.9223
4,NM_013290,331.9910
6,NM_001625,4769.1790
7,lincRNA:chr7:226042-232442_R,2311.7730
11,NM_005937,2229.0950


## 11. 발현값에 log2 변환 적용

마이크로어레이 발현값은 동적 범위가 넓어 log2 변환 후 분포가 안정화됩니다.  
`log2(x)`는 x>0일 때만 정의되므로, 0 이하 값은 변환 전 제거합니다.

In [12]:
# ------------------------------------------------------------
# 11) log2 변환
# ------------------------------------------------------------
n_nonpositive = (slim["gProcessedSignal"] <= 0).sum()
print(f"[11] log2 변환 전 0 이하 신호 개수: {n_nonpositive:,}")

log2_df = slim[slim["gProcessedSignal"] > 0].copy()
log2_df["gProcessedSignal_log2"] = np.log2(log2_df["gProcessedSignal"])

# 원본 선형 스케일 컬럼은 제거하고 log2 값만 남김 (ML 입력용)
final_df = log2_df[["SystematicName", "gProcessedSignal_log2"]].rename(
    columns={"gProcessedSignal_log2": "gProcessedSignal"}
)

print("[11] log2 변환 적용 완료 (컬럼명 gProcessedSignal = log2 값)")
report_size("11_log2_transformed", final_df)
display(final_df.head(5))
print("[11] 발현값 요약 통계:")
display(final_df["gProcessedSignal"].describe())

[11] log2 변환 전 0 이하 신호 개수: 0
[11] log2 변환 적용 완료 (컬럼명 gProcessedSignal = log2 값)
[11_log2_transformed] 데이터 크기: 34,335 rows × 2 columns


,SystematicName,gProcessedSignal
3,NM_144987,8.266423
4,NM_013290,8.375000
6,NM_001625,12.219525
7,lincRNA:chr7:226042-232442_R,11.174784
11,NM_005937,11.122242


[11] 발현값 요약 통계:


count    34335.000000
mean         9.993296
std          2.198269
min          6.708631
25%          8.258565
50%          9.720193
75%         11.346793
max         18.933024
Name: gProcessedSignal, dtype: float64

## 12. 최종 결과를 CSV로 저장

In [13]:
# ------------------------------------------------------------
# 12) CSV 저장
# ------------------------------------------------------------
final_df.to_csv(CSV_OUT, index=False, encoding="utf-8")
print(f"[12] CSV 저장 완료: {CSV_OUT.resolve()}")
report_size("12_saved_csv", final_df)

[12] CSV 저장 완료: C:\MyCursorLab\01_바이오 데이터 이해 및 LLM 기반 전처리\GSM827073_preprocessed.csv
[12_saved_csv] 데이터 크기: 34,335 rows × 2 columns


## 13–14. 전처리 전후 비교 및 단계별 처리 요약

In [14]:
# ------------------------------------------------------------
# 13) 전처리 전후 데이터 개수 비교
# 14) 각 단계 처리 내용 출력
# ------------------------------------------------------------
summary = pd.DataFrame(
    {
        "단계": list(step_sizes.keys()),
        "probe_수": list(step_sizes.values()),
    }
)
summary["이전_대비_감소"] = summary["probe_수"].diff().fillna(0).astype(int)

n_before = step_sizes.get("3_all_features", n_features_total)
n_after = len(final_df)

print("=" * 60)
print("[13] 전처리 전후 비교")
print("=" * 60)
print(f"  전처리 전 (전체 Feature) : {n_before:,}")
print(f"  전처리 후 (최종 CSV)     : {n_after:,}")
print(f"  제거된 Feature 수        : {n_before - n_after:,}")
print(f"  잔존 비율                : {n_after / n_before * 100:.2f}%")
print()
print("=" * 60)
print("[14] 단계별 처리 요약")
print("=" * 60)
display(summary)

print("\n[처리 파이프라인 요약]")
print("  1. 압축 해제 TXT 읽기")
print("  2. FEATURES/DATA → DataFrame")
print("  3. 전체 Feature 수 확인")
print("  4. ControlType 분포 확인")
print("  5. biological probe (ControlType==0)")
print("  6. gProcessedSignal 결측 제거")
print("  7. gIsPosAndSignif==1")
print("  8. gIsWellAboveBG==1")
print("  9. 저발현(하위 25%) 필터")
print(" 10. SystematicName + gProcessedSignal")
print(" 11. log2 변환")
print(" 12. CSV 저장")
print(f"\n최종 파일: {CSV_OUT.resolve()}")

[13] 전처리 전후 비교
  전처리 전 (전체 Feature) : 62,976
  전처리 후 (최종 CSV)     : 34,335
  제거된 Feature 수        : 28,641
  잔존 비율                : 54.52%

[14] 단계별 처리 요약


,단계,probe_수,이전_대비_감소
0,1_raw_read,62976,0
1,2_features_df,62976,0
2,3_all_features,62976,0
3,4_before_control_filter,62976,0
4,5_biological_probes,58717,-4259
5,6_drop_missing_signal,58717,0
6,7_pos_and_signif,53565,-5152
7,8_well_above_bg,45780,-7785
8,9_low_expression_filtered,34335,-11445
9,10_selected_columns,34335,0



[처리 파이프라인 요약]
  1. 압축 해제 TXT 읽기
  2. FEATURES/DATA → DataFrame
  3. 전체 Feature 수 확인
  4. ControlType 분포 확인
  5. biological probe (ControlType==0)
  6. gProcessedSignal 결측 제거
  7. gIsPosAndSignif==1
  8. gIsWellAboveBG==1
  9. 저발현(하위 25%) 필터
 10. SystematicName + gProcessedSignal
 11. log2 변환
 12. CSV 저장

최종 파일: C:\MyCursorLab\01_바이오 데이터 이해 및 LLM 기반 전처리\GSM827073_preprocessed.csv
